In [ ]:
import pandas as pd
import psycopg
import json
from pgvector.psycopg import register_vector

print("Lese Parquet-Dateien ein...")
df_hybrid       = pd.read_parquet("parquet/multilingual-e5-large/embeddings_docling_hybrid.parquet")
df_hierarchical = pd.read_parquet("parquet/multilingual-e5-large/embeddings_docling_hierarchical.parquet")
df_mdheader     = pd.read_parquet("parquet/multilingual-e5-large/embeddings_langchain_mdheader.parquet")
df_recursive    = pd.read_parquet("parquet/multilingual-e5-large/embeddings_langchain_recursive.parquet")

# ---------------------------------------------------------
# 2. FILTERN: Nur Dokumente behalten, die ÜBERALL existieren
# ---------------------------------------------------------
gemeinsame_ids = (
    set(df_hybrid["doc_id"]) & 
    set(df_hierarchical["doc_id"]) & 
    set(df_mdheader["doc_id"]) & 
    set(df_recursive["doc_id"])
)

print(f"📊 Gefundene gemeinsame Dokumente: {len(gemeinsame_ids)}")
print(f"Gemeinsame Dokument-IDs: {list(gemeinsame_ids)}")

# Filtern der DataFrames auf die gemeinsamen Dokument-IDs
df_hybrid       = df_hybrid[df_hybrid["doc_id"].isin(gemeinsame_ids)]
df_hierarchical = df_hierarchical[df_hierarchical["doc_id"].isin(gemeinsame_ids)]
df_mdheader     = df_mdheader[df_mdheader["doc_id"].isin(gemeinsame_ids)]
df_recursive    = df_recursive[df_recursive["doc_id"].isin(gemeinsame_ids)]

# ---------------------------------------------------------
# 3. DATENBANK-UPLOAD (Hilfsfunktion)
# ---------------------------------------------------------
def lade_df_in_tabelle(df, tabellen_name, conn):
    if df.empty:
        print(f"⚠️ {tabellen_name} ist leer, überspringe...")
        return
        
    data_to_insert = []
    for _, row in df.iterrows():
        metadata_json = json.dumps({}) 
        
        data_to_insert.append((
            row["doc_id"], 
            row["chunk_index"], 
            row["content"], 
            row["token_count"], 
            metadata_json,
            row["embedding"]
        ))
        
    query = f"""
        INSERT INTO {tabellen_name} (doc_id, chunk_index, content, token_count, metadata, embedding)
        VALUES (%s, %s, %s, %s, %s, %s);
    """
    
    with conn.cursor() as cur:
        print(f"🚀 Lade {len(data_to_insert)} Chunks in die Tabelle '{tabellen_name}'...")
        cur.executemany(query, data_to_insert)
        conn.commit()
    print(f"✅ {tabellen_name} erfolgreich befüllt!")

# ---------------------------------------------------------
# 4. UPLOAD STARTEN
# ---------------------------------------------------------
DB_CONNECTION = "dbname=chunks_evaluation user=admin password=admin host=localhost port=59101"

print("\nVerbinde mit lokaler Datenbank...")
conn = psycopg.connect(DB_CONNECTION)
register_vector(conn)

lade_df_in_tabelle(df_hybrid, "chunks_docling_hybrid", conn)
lade_df_in_tabelle(df_hierarchical, "chunks_docling_hierarchical", conn)
lade_df_in_tabelle(df_recursive, "chunks_langchain_recursive", conn)
lade_df_in_tabelle(df_mdheader, "chunks_langchain_mdheader", conn)

conn.close()
print("\n🎉 BÄM! Alle Daten sind sicher und fair in deiner Datenbank!")

Lese Parquet-Dateien ein...
📊 Gefundene gemeinsame Dokumente: 90

Verbinde mit lokaler Datenbank...
🚀 Lade 6082 Chunks in die Tabelle 'chunks_docling_hybrid'...
✅ chunks_docling_hybrid erfolgreich befüllt!
🚀 Lade 25601 Chunks in die Tabelle 'chunks_docling_hierarchical'...
✅ chunks_docling_hierarchical erfolgreich befüllt!
🚀 Lade 6607 Chunks in die Tabelle 'chunks_langchain_recursive'...
✅ chunks_langchain_recursive erfolgreich befüllt!
🚀 Lade 6942 Chunks in die Tabelle 'chunks_langchain_mdheader'...
✅ chunks_langchain_mdheader erfolgreich befüllt!

🎉 BÄM! Alle Daten sind sicher und fair in deiner Datenbank!
